# Importing publications

In [ ]:
import pandas as pd
import numpy as np
import requests
from owlready2 import *
import rdflib
import re
from collections import Counter

## DLS

In [103]:
# Import and concat all DLS publications
pathname = '/Users/fdp54928/Library/CloudStorage/OneDrive-Nexus365/GitHub Repositories/synchrotron-proposals-topic-classification/Datasets'
pub_dls=pd.concat([
pd.read_csv(pathname+'/DLS/DLS annual review highlight.csv'),
pd.read_csv(pathname+'/DLS/DLS book chapter.csv'),
pd.read_csv(pathname+'/DLS/DLS conference paper.csv'),
pd.read_csv(pathname+'/DLS/DLS editor note.csv'),
pd.read_csv(pathname+'/DLS/DLS journal paper (2002-2010).csv'),
pd.read_csv(pathname+'/DLS/DLS journal paper (2011-2020).csv'),
pd.read_csv(pathname+'/DLS/DLS journal paper (2021-2024).csv'),
pd.read_csv(pathname+'/DLS/DLS magazine article.csv'),
pd.read_csv(pathname+'/DLS/DLS poster.csv'),
pd.read_csv(pathname+'/DLS/DLS report.csv'),
pd.read_csv(pathname+'/DLS/DLS science highlight.csv'),
pd.read_csv(pathname+'/DLS/DLS thesis.csv'),
pd.read_csv(pathname+'/DLS/DLS all publications 2024.csv'),
pd.read_csv(pathname+'/DLS/DLS all publications 2025.csv')
],ignore_index=True)

# Add 'Facility repository' column
pub_dls['Facility repository']='DLS'

# Remove entries with no DOI and duplicates based on DOI
pub_dls = pub_dls[pub_dls['DOI'].notnull()].drop_duplicates(subset=['DOI'], keep='first')

In [66]:
pub_dls.columns

Index(['Publications Database Id', 'DOI', 'PMID', 'ISI ID', 'Title of Paper',
       'Authors', 'Industrial Partner Is Co-author?', 'Publication State',
       'Date Published', 'Diamond Proposal Number', 'Publication Type',
       'Title of Journal', 'Journal Volume', 'Journal Pages',
       'Title of Conference', 'Peer Reviewed', 'Magazine Title',
       'Uses Synchrotron, EM or Offline lab Data?', 'Data From Diamond?',
       'Beamlines', 'Additional Facilities If Data From Diamond',
       'Facility If Data Not From Diamond', 'Subject Areas', 'Technical Areas',
       'Keywords', 'Diamond Keywords', 'Discipline/Technical Tags', 'ISBN',
       'Book Chapter', 'Added On', 'Facility repository'],
      dtype='object')

In [5]:
len(pub_dls)

16186

## ESRF

In [176]:
filepath = '/Users/fdp54928/Library/CloudStorage/OneDrive-Nexus365/GitHub Repositories/synchrotron-proposals-topic-classification/Datasets/ESRF/Checkpoint_data/Publications_ESRF.csv'
pub_esrf = pd.read_csv(filepath,encoding='utf-8')

In [188]:
pub_esrf[pub_esrf['Beamline(s)'].notna()]['Beamline(s)'].str.contains('/',regex=False).sum()

np.int64(6115)

In [190]:
pub_esrf[pub_esrf['Beamline(s)'].notna()]

,Document type,Open Access,Number,Authors countries,Authors,Title,Journal title,Volume,Pages,Year,Beamline(s),Facility used,Proposal number,DOI,DOI added,WoS Number,Type,Facility repository
0,Journal article,OA,ESRF25KO101,The Netherlands/France (ESRF),"Komarova T.Yu., Zinn T., Narayanan T., Petukho...",Microtube self-assembly leads to conformationa...,Journal of Colloid and Interface Science,677,781-789,2025.0,ID2,ESRF (Grenoble),"SC-4987, SC-5177",10.1016/j.jcis.2024.08.003,NaN,NaN,1,ESRF
1,Journal article,OA,ESRF24AD472,Germany/France (ESRF),"Adler P., Medvedev S.A., Mu Q., Bessas D., Chu...",Electronic and magnetic phase diagram of Sr2Fe...,Physical Review B,110,054444-1-054444-12,2024.0,ID18,ESRF (Grenoble),"HC-4292, HC-4728",10.1103/PhysRevB.110.054444,NaN,NaN,1,ESRF
2,Journal article,NaN,ESRF24AG344,France (not ESRF)/USA/France (ESRF)/UK,"Agafonov A., Pineda-Romero N., Witman M., Nass...",Destabilizing high-capacity high entropy hydri...,Acta Materialia,276,120086-1-120086-10,2024.0,ID15A,ESRF (Grenoble),NaN,10.1016/j.actamat.2024.120086,NaN,NaN,1,ESRF
3,Journal article,NaN,ESRF24AG388,UK/Italy/France (not ESRF)/France (ESRF)/Germa...,"Agrestini S., Borgatti F., Florio P., Frassine...",Origin of magnetism in a supposedly nonmagneti...,Physical Review Letters,133,066501-1-066501-7,2024.0,ID20,ESRF (Grenoble)/DIAMOND (Didcot)/PETRA III,"HC-4277, HC-4912",10.1103/PhysRevLett.133.066501,NaN,NaN,1,ESRF
4,Journal article,OA,ESRF24AK362,Germany/Sweden/UK/France (ESRF),"Akbar F., Aslandukova A., Yin Y., Aslandukov A...",High-pressure dysprosium carbides containing c...,Carbon,228,119374-1-119374-8,2024.0,ID11/ID15B,ESRF (Grenoble),NaN,10.1016/j.carbon.2024.119374,NaN,NaN,1,ESRF
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
47276,Journal article,NaN,ESRF04SA7022,Japan,"Sakabe N., Sakabe K., Sasaki K.",Conceptual design of novel IP-conveyor-belt We...,Journal of Synchrotron Radiation,11,12-16,2004.0,ID9,NaN,NaN,NaN,NaN,NaN,4,ESRF
47486,Book chapter,NaN,ESRF02GI7031,France (not ESRF)/USA,"Gibbs D., Hill J.P., Vettier C.",New directions in X-ray magnetic scattering,NaN,NaN,NaN,NaN,ID20,NaN,NaN,NaN,NaN,NaN,4,ESRF
47531,Journal article,NaN,ESRF06TE7020,Switzerland,"Tedesco E., Giron D., Pfeffer S.",Crystal structure elucidation and morphology s...,CrystEngComm,4,393-400,2002.0,BM16,NaN,NaN,10.1039/b202427f,NaN,NaN,4,ESRF
47680,Journal article,NaN,ESRF04CH7063,Denmark,"Christensen A.N., Chevallier M.A., Skibsted J....",Synthesis and characterization of basic bismut...,Journal of the Chemical Society Dalton Transac...,NaN,265-270,2000.0,BM1A,ESRF (Grenoble),NaN,10.1039/a908055d,NaN,WOS:000084955900008,4,ESRF


# Loading PaNET ontology

In [6]:
# Load the ontology

ontology_path='/Users/fdp54928/Documents/PaNET-classifier/Data/owlapi.xrdf'
onto=get_ontology(ontology_path).load()

In [7]:
# Run reasoner

with onto:
    sync_reasoner()  # Runs reasoning and updates inferred relationships

* Owlready2 * Running HermiT...
    java -Xmx2000M -cp /Users/fdp54928/Documents/PaNET-classifier/.venv/lib/python3.13/site-packages/owlready2/hermit:/Users/fdp54928/Documents/PaNET-classifier/.venv/lib/python3.13/site-packages/owlready2/hermit/HermiT.jar org.semanticweb.HermiT.cli.CommandLine -c -O -D -I file:////var/folders/jt/qsn_d43943dbh0h9ngcrb3n80000gq/T/tmpz_hd8i9c
* Owlready2 * HermiT took 1.0086021423339844 seconds
* Owlready * Reparenting PaNET.PaNET1001000: {PaNET.PaNET1000000, owl.ObjectProperty, owl.AsymmetricProperty, owl.IrreflexiveProperty} => {PaNET.PaNET1000000}
* Owlready * Reparenting PaNET.PaNET1004000: {PaNET.PaNET1000000, owl.ObjectProperty, owl.AsymmetricProperty, owl.IrreflexiveProperty} => {PaNET.PaNET1000000}
* Owlready * Reparenting PaNET.PaNET1002000: {PaNET.PaNET1000000, owl.ObjectProperty, owl.AsymmetricProperty, owl.IrreflexiveProperty} => {PaNET.PaNET1000000}
* Owlready * Reparenting PaNET.PaNET1003000: {PaNET.PaNET1000000, owl.ObjectProperty, owl.Asym

In [8]:
onto_ls = list(onto.classes())
panet00001 = onto.search(iri='http://purl.org/pan-science/PaNET/PaNET00001')[0]
technique_terms = [cls for cls in panet00001.descendants() if cls.iri!='https://www.wikidata.org/wiki/Q133900']
technique_label = [cls.label[0].strip().lower() for cls in technique_terms]
technique_iri = [cls.iri for cls in technique_terms]
technique_altLabel = [cls.altLabel for cls in technique_terms]

print('Number of technique classes:',len(technique_iri))

Number of technique classes: 377


# Mapping technical tags to PaNET terms

In [12]:
pathname = '/Users/fdp54928/Documents/PaNET-classifier/Data/PaNET_mapping.xlsx'
panet_mapping = pd.read_excel(pathname)


In [13]:
panet_mapping

,Technical tags,PaNET ID,PaNET IRI,Comments
0,Diffraction,http://purl.org/pan-science/PaNET/PaNET01022,diffraction,NaN
1,Imaging,http://purl.org/pan-science/PaNET/PaNET01106,imaging,NaN
2,Microscopy,http://purl.org/pan-science/PaNET/PaNET01069,microscopy,NaN
3,Scattering,http://purl.org/pan-science/PaNET/PaNET00200,scattering technique,NaN
4,Spectroscopy,http://purl.org/pan-science/PaNET/PaNET01125,spectroscopy,NaN
...,...,...,...,...
106,X-ray Absorption Near Edge Structure (XANES),http://purl.org/pan-science/PaNET/PaNET01199,x-ray absorption near edge structure,This PaNET term has both the altLabels NEXAFS ...
107,Hard X-ray Photoelectron Spectroscopy (HAXPES),http://purl.org/pan-science/PaNET/PaNET01103,hard x-ray photoelectron spectroscopy,NaN
108,Near Ambient Pressure XPS (NAP-XPS),http://purl.org/pan-science/PaNET/PaNET01218,ambient pressure x-ray photoelectron spectroscopy,Near ambient and ambient pressure XPS seem to ...
109,Soft X-ray Ptychography,NaN,NaN,NaN


In [67]:
def extract_panet_terms(row, technical_tags_ls):
    row_set = set(term.strip() for term in row.split(','))
    tech_set = set(term.strip() for term in technical_tags_ls)
    found_techniques = list(row_set.intersection(tech_set))
    return found_techniques

In [68]:
extracted_techniques = pub_dls['Discipline/Technical Tags'].fillna('').apply(lambda x: extract_panet_terms(x, panet_mapping['Technical tags'].to_list()))

In [69]:
# Check which technical tags in PaNET mapping are not found in publications

# Flatten the list of extracted techniques
ls = []
for term in extracted_techniques:
    ls.extend(term)
    x=set(ls)

# Set of technical tags in PaNET mapping that are not found in publications
y = set(panet_mapping['Technical tags'].to_list())
print('Technical tags in PaNET mapping not found in publications:')
x.symmetric_difference(y)

Technical tags in PaNET mapping not found in publications:


{'Anomalous Diffraction',
 'Anomalous Scattering',
 'Correlative Light Electron Microscopy (CLEM)',
 'Cryo Focused Ion Beam Scanning Electron Microscopy (FIBSEM)',
 'Diffracted X-ray Tracking (DXT)',
 'Fixed Target Serial Synchrotron Crystallography (FT-SSX)',
 'In Situ Diffraction',
 'Infrared Microscopy',
 'Infrared Nanospectroscopy Imaging',
 'Lipidic Cubic Phase Serial Synchrotron Crystallography (LCP-SSX)',
 'Magnetic X-ray Tomography (MXT)',
 'Molecular Replacement',
 'Nano Infrared Spectroscopy',
 'Photo Crystallography',
 'Serial Femtosecond Crystallography (SFX)',
 'Time Resolved Serial Femtosecond Crystallography (TR-SFX)',
 'UV Circular Dichroism Imaging',
 'UV Synchrotron Radiation Circular Dichroism (SRCD)',
 'Ultra Small Angle X-ray Scattering (USAXS)'}

In [70]:
print('Number of publications with at least one extracted technique:')
extracted_techniques.apply(lambda x: len(x)!=0).value_counts()

Number of publications with at least one extracted technique:


Discipline/Technical Tags
True     13499
False     2687
Name: count, dtype: int64

In [71]:
extracted_techniques

62                                                      []
63                                                      []
64                                                      []
65                 [Spectroscopy, Circular Dichroism (CD)]
66       [Cryo Electron Microscopy (Cryo EM), Electron ...
                               ...                        
18704                                [Imaging, Tomography]
18705    [Diffraction, Macromolecular Crystallography (...
18706    [Cryo Electron Microscopy (Cryo EM), Electron ...
18707    [Diffraction, Macromolecular Crystallography (...
18709    [Small Angle X-ray Scattering (SAXS), Scattering]
Name: Discipline/Technical Tags, Length: 16186, dtype: object

In [104]:
pub_dls = pd.concat([pub_dls, extracted_techniques.rename('Extracted Technical Tags')], axis=1)

In [105]:
df = pub_dls[pub_dls['Extracted Technical Tags'].apply(lambda x: len(x)!=0)]

Map the technical tags to PaNET IDs

In [107]:
def map_technical_tags_to_panet_ids(row, mapping_dict):
    panet_terms = []
    for tag in row:
        panet_term = mapping_dict[tag.strip()]
        if str(panet_term) != 'nan':
            panet_terms.append(panet_term) 
    return panet_terms

In [108]:
mapping_dict = dict(zip(panet_mapping['Technical tags'], panet_mapping['PaNET ID']))
extracted_panet_ids = df['Extracted Technical Tags'].apply(lambda x: map_technical_tags_to_panet_ids(x, mapping_dict))
df = pd.concat([df, extracted_panet_ids.rename('Extracted PaNET IDs')], axis=1)

In [ ]:
def get_hierarchy_terms(panet_id, ontology):
    cls = ontology.search(iri=panet_id)[0]
    hierarchy_terms = [ancestor.label[0].strip() for ancestor in cls.ancestors() if ancestor.label and ancestor.label[0].strip() != 'Thing']
    return hierarchy_terms

In [78]:
# df = df[df['Beamlines'].notna()]

In [79]:
# df['Beamlines'].groupby(df['Beamlines']).count()

In [193]:
df['Extracted PaNET Terms']

65                              [spectroscopy]
66                                [microscopy]
68                       [tomography, imaging]
72                               [diffraction]
77       [spectroscopy, infrared spectroscopy]
                         ...                  
18701                            [diffraction]
18704                    [tomography, imaging]
18705                            [diffraction]
18706                             [microscopy]
18707                            [diffraction]
Name: Extracted PaNET Terms, Length: 12074, dtype: object

In [ ]:
# def extract_beamlines(row):
#     row_set = set(beamline.strip() for beamline in row.split(','))
#     return list(row_set)

In [ ]:
# beamline_set = set(y for x in df['Beamlines'].apply(extract_beamlines) for y in x)

In [ ]:
# df_unique_beamline = df[~df['Beamlines'].str.contains(',')]

In [ ]:
# beamline_dict = {}

# for beamline in beamline_set:
#     beamline_dict[beamline] = df_unique_beamline[df_unique_beamline['Beamlines'].str.contains(beamline, regex=False)]['Extracted PaNET Terms'].explode().to_list()

In [80]:
# beamline_dict

In [81]:
# df_unique_beamline['Beamlines'].value_counts()

In [82]:
Counter([y for x in extracted_techniques for y in x])

Counter({'Diffraction': 8007,
         'Macromolecular Crystallography (MX)': 5143,
         'Spectroscopy': 2618,
         'Scattering': 1731,
         'X-ray Absorption Spectroscopy (XAS)': 1424,
         'Small Angle X-ray Scattering (SAXS)': 1166,
         'Imaging': 1033,
         'X-ray Powder Diffraction': 1003,
         'Single Crystal X-ray Diffraction (SXRD)': 913,
         'Microscopy': 824,
         'Electron Microscopy (EM)': 764,
         'Tomography': 620,
         'Extended X-ray Absorption Fine Structure (EXAFS)': 519,
         'X-ray Absorption Near Edge Structure (XANES)': 476,
         'Circular Dichroism (CD)': 462,
         'Cryo Electron Microscopy (Cryo EM)': 397,
         'X-ray Photoelectron Spectroscopy (XPS)': 320,
         'Angle Resolved Photoemission Spectroscopy (ARPES)': 285,
         'Wide Angle X-ray Scattering (WAXS)': 261,
         'X-ray Fluorescence (XRF)': 258,
         'Infrared Spectroscopy': 197,
         'X-ray Magnetic Circular Dichroism (XM